# Purpose
I will use the csv file from City of Chicago's Food Inspection page, combing it with the insights from the Excel audit to create a SQLite database and then apply some simple standardizations (using UPPER and TRIM to standardize each column, and COALESCE missing values into a single "Unknown" category)

I also want to engineer a few features: Year and Month.

# Outline
* use Dataframe.explode to split up the text found in violations into its own row. Within each text, each violation, prepended by a code, is split by ' | '.
*  Create a three table data model using pandas:
    * inspections: Inspection ID (PK), License # (FK), Inspection Date, Inspection Type, Results, Risk, Facility Type, Zip, Address
    * violations: Violation ID (PK), Inspection ID (FK), Violation Code, Violation Text
* Do simple standardization: UPPER and TRIM on DBA Name, Facility Type, and Address
* COALESCE missing values in ZIP Codes, and Risk to "Unknown." 
* Feature Engineer Year and Month
* 

Originally, I had a separate facilities tables but it turns out that license #s were not unique to facilities which made it cumbersome to work with, and created duplicate records on joins. So, I decided to merge relevant facilities table values with the inspections table. License # is not used in the table so I do not need to COALESCE License # to "Unknown", including the 0 values.

In [1]:
import sqlite3 as sql
import pandas as pd

In [2]:
con = sql.connect("food_inspections.db")
cur = con.cursor()
df = pd.read_csv("data\\raw\\Food_Inspections_20260613.csv")

In [3]:
df.columns = [
    "inspection_id","dba_name","aka_name","license_id","facility_type","risk","address","city","state","zip",
    "inspection_date","inspection_type","results","violations","latitude","longitude","location"]

In [4]:
df.head()

,inspection_id,dba_name,aka_name,license_id,facility_type,risk,address,city,state,zip,inspection_date,inspection_type,results,violations,latitude,longitude,location
0,2638434,LITTLE VILAGE NURSING AND REHABILITATION CENTE...,LITTLE VILAGE NURSING AND REHABILITATION CENTE...,2570288.0,Long Term Care,Risk 1 (High),2320 S LAWNDALE AVE,CHICAGO,IL,60623.0,06/12/2026,Canvass,Pass,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,41.849156,-87.717512,"(41.84915619738583, -87.71751181796799)"
1,2638421,TAWNEY COFFEE LLC,TAWNEY COFFEE,3040996.0,Restaurant,Risk 2 (Medium),3055-3057 W POPE JOHN PAUL II DR,CHICAGO,IL,60632.0,06/12/2026,License Re-Inspection,Pass,NaN,41.815329,-87.701541,"(41.81532948609713, -87.70154086004867)"
2,2638431,BORELLI'S,BORELLI'S,2121437.0,Restaurant,Risk 1 (High),2124 W LAWRENCE AVE,CHICAGO,IL,60625.0,06/12/2026,Complaint Re-Inspection,Pass,NaN,41.968814,-87.682645,"(41.96881436962451, -87.68264478410583)"
3,2638417,PINK HEALTH NUTRITION,PINK HEALTH NUTRITION,3077889.0,NaN,NaN,3015 W 63rd ST,CHICAGO,IL,60629.0,06/12/2026,License,Not Ready,NaN,41.778924,-87.698964,"(41.778924105979264, -87.69896362746249)"
4,2638381,TAQUERIA LOS GALLOS INC,TAQUERIA LOS GALLOS,1044860.0,Restaurant,Risk 1 (High),4209-4211 W 26TH ST,CHICAGO,IL,60623.0,06/11/2026,Canvass Re-Inspection,Pass,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,41.844070,-87.729807,"(41.844069583410565, -87.72980747367433)"


In [5]:
inspections_df = df[["inspection_id", "dba_name", "facility_type", "address", "zip", "inspection_date", "inspection_type", "results", "risk"]]
inspections_df.head()

,inspection_id,dba_name,facility_type,address,zip,inspection_date,inspection_type,results,risk
0,2638434,LITTLE VILAGE NURSING AND REHABILITATION CENTE...,Long Term Care,2320 S LAWNDALE AVE,60623.0,06/12/2026,Canvass,Pass,Risk 1 (High)
1,2638421,TAWNEY COFFEE LLC,Restaurant,3055-3057 W POPE JOHN PAUL II DR,60632.0,06/12/2026,License Re-Inspection,Pass,Risk 2 (Medium)
2,2638431,BORELLI'S,Restaurant,2124 W LAWRENCE AVE,60625.0,06/12/2026,Complaint Re-Inspection,Pass,Risk 1 (High)
3,2638417,PINK HEALTH NUTRITION,NaN,3015 W 63rd ST,60629.0,06/12/2026,License,Not Ready,NaN
4,2638381,TAQUERIA LOS GALLOS INC,Restaurant,4209-4211 W 26TH ST,60623.0,06/11/2026,Canvass Re-Inspection,Pass,Risk 1 (High)


In [6]:
violations_df = df[["inspection_id", "violations"]]
violations_df.count()

inspection_id    311775
violations       224138
dtype: int64

In [7]:
violations_df["violations"] = violations_df["violations"].fillna("").str.split(r"\s*\|\s*")

In [8]:
violations_df = violations_df.explode("violations")

In [9]:
violations_df.count()

inspection_id    1099664
violations       1099664
dtype: int64

In [10]:
violations_df["violation_code"] = violations_df["violations"].str.extract(r"^(\d+)")

In [11]:
violations_df["violation_text"] = violations_df["violations"]
violations_df.drop(columns="violations", inplace=True)

In [12]:
violations_df.reset_index(names="violation_id")

,violation_id,inspection_id,violation_code,violation_text
0,0,2638434,47,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...
1,1,2638421,NaN,
2,2,2638431,NaN,
3,3,2638417,NaN,
4,4,2638381,16,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...
...,...,...,...,...
1099659,311774,67738,18,18. NO EVIDENCE OF RODENT OR INSECT OUTER OPEN...
1099660,311774,67738,32,32. FOOD AND NON-FOOD CONTACT SURFACES PROPERL...
1099661,311774,67738,34,"34. FLOORS: CONSTRUCTED PER CODE, CLEANED, GOO..."
1099662,311774,67738,35,"35. WALLS, CEILINGS, ATTACHED EQUIPMENT CONSTR..."


In [13]:
inspections_df.to_sql(
    "inspections",
    con,
    if_exists="replace",
    index=False
)

311775

In [14]:
violations_df.to_sql(
    "violations",
    con,
    if_exists="replace",
    index=False
)

1099664

## SQL Cleaning

In [15]:
con.execute("""CREATE VIEW violation_analysis AS
SELECT 
    v.inspection_id,
    v.violation_code,
    v.violation_text,
    i.inspection_date,
    i.results,
    COALESCE(i.risk, 'Unknown') AS risk,
    UPPER(TRIM(i.dba_name)) AS dba_name,
    COALESCE(UPPER(TRIM(i.facility_type)), 'Unknown') AS facility_type,
    UPPER(TRIM(i.address)) AS address,
    COALESCE(i.zip, 'Unknown') AS zip
FROM violations AS v
JOIN inspections AS i
    ON v.inspection_id = i.inspection_id;""")


# SQL Exploration

In [16]:
query = """SELECT 
    violation_code,
    COUNT(*) AS violation_count
FROM violation_analysis
WHERE violation_code <> 'None'
GROUP BY violation_code
ORDER BY violation_count DESC;
"""
cur.execute(query)
cur.fetchall()

[('55', 82926),
 ('38', 76851),
 ('34', 71800),
 ('33', 67390),
 ('35', 65915),
 ('32', 55368),
 ('41', 40366),
 ('47', 38201),
 ('36', 35675),
 ('49', 30011),
 ('10', 28852),
 ('51', 27768),
 ('3', 27482),
 ('56', 24103),
 ('37', 20505),
 ('40', 20273),
 ('58', 19901),
 ('5', 19329),
 ('18', 17003),
 ('53', 16989),
 ('2', 16898),
 ('30', 16302),
 ('57', 15093),
 ('16', 14061),
 ('43', 12356),
 ('39', 12223),
 ('45', 11269),
 ('21', 10820),
 ('31', 9941),
 ('48', 8814),
 ('22', 7566),
 ('29', 6942),
 ('23', 6464),
 ('42', 6461),
 ('60', 6436),
 ('25', 6073),
 ('54', 6008),
 ('1', 5717),
 ('44', 4809),
 ('19', 4350),
 ('11', 3776),
 ('8', 3524),
 ('24', 3323),
 ('12', 3065),
 ('6', 2507),
 ('14', 2349),
 ('9', 2299),
 ('52', 2286),
 ('28', 1930),
 ('61', 1835),
 ('59', 1779),
 ('50', 1481),
 ('26', 1440),
 ('64', 1038),
 ('13', 867),
 ('15', 601),
 ('70', 595),
 ('4', 531),
 ('27', 407),
 ('46', 266),
 ('20', 233),
 ('62', 209),
 ('7', 165),
 ('17', 162),
 ('63', 48)]

In [17]:
query = """SELECT
    facility_type,
    violation_code,
    COUNT(*) AS violation_count
FROM violation_analysis
WHERE violation_code <> 'None' AND facility_type <> 'None'
GROUP BY facility_type, violation_code
ORDER BY facility_type, violation_count DESC;
"""
cur.execute(query)
cur.fetchmany(25)

[('(CONVENIENCE STORE)', '34', 1),
 ('(CONVENIENCE STORE)', '32', 1),
 ('(GAS STATION)', '38', 2),
 ('(GAS STATION)', '41', 1),
 ('(GAS STATION)', '35', 1),
 ('(GAS STATION)', '34', 1),
 ('(GAS STATION)', '3', 1),
 ('(GAS STATION)', '2', 1),
 ('(GAS STATION)', '19', 1),
 ('1005 NURSING HOME', '37', 1),
 ('1005 NURSING HOME', '35', 1),
 ('1005 NURSING HOME', '34', 1),
 ('1005 NURSING HOME', '33', 1),
 ('1005 NURSING HOME', '24', 1),
 ('1005 NURSING HOME', '19', 1),
 ('1005 NURSING HOME', '18', 1),
 ('1023', '36', 7),
 ('1023', '34', 5),
 ('1023', '38', 3),
 ('1023', '32', 3),
 ('1023', '40', 2),
 ('1023', '35', 2),
 ('1023', '33', 2),
 ('1023', '24', 2),
 ('1023', '30', 1)]

In [18]:
query = """SELECT
    zip,
    violation_code,
    COUNT(*) AS violation_count
FROM violation_analysis
WHERE violation_code <> 'None' AND zip <> 'Unknown'
GROUP BY zip, violation_code
ORDER BY zip, violation_count DESC;
"""
cur.execute(query)
cur.fetchmany(25)

[(10014.0, '58', 1),
 (10014.0, '3', 1),
 (46319.0, '5', 1),
 (46319.0, '37', 1),
 (46322.0, '56', 1),
 (46410.0, '37', 7),
 (46410.0, '38', 4),
 (46410.0, '57', 3),
 (46410.0, '50', 1),
 (46410.0, '48', 1),
 (46410.0, '44', 1),
 (46410.0, '3', 1),
 (46410.0, '10', 1),
 (53061.0, '57', 1),
 (53061.0, '3', 1),
 (54971.0, '60', 1),
 (54971.0, '58', 1),
 (54971.0, '57', 1),
 (60007.0, '32', 5),
 (60007.0, '34', 3),
 (60007.0, '38', 2),
 (60007.0, '9', 1),
 (60007.0, '33', 1),
 (60007.0, '30', 1),
 (60007.0, '18', 1)]

In [19]:
query = """SELECT
    violation_code,
    COUNT(*) AS fail_count
FROM violation_analysis
WHERE violation_code <> 'None'
GROUP BY violation_code
ORDER BY fail_count DESC;
"""
cur.execute(query)
cur.fetchmany(25)

[('55', 82926),
 ('38', 76851),
 ('34', 71800),
 ('33', 67390),
 ('35', 65915),
 ('32', 55368),
 ('41', 40366),
 ('47', 38201),
 ('36', 35675),
 ('49', 30011),
 ('10', 28852),
 ('51', 27768),
 ('3', 27482),
 ('56', 24103),
 ('37', 20505),
 ('40', 20273),
 ('58', 19901),
 ('5', 19329),
 ('18', 17003),
 ('53', 16989),
 ('2', 16898),
 ('30', 16302),
 ('57', 15093),
 ('16', 14061),
 ('43', 12356)]

In [20]:
query = """SELECT * FROM violation_analysis"""
cur.execute(query)
cleaned_df = pd.DataFrame(cur.fetchall(), columns=["inspection_id", "violation_code", "violation_text", "inspection_date", "results", "risk", "dba_name", "facility_type", "address", "zip"])

In [21]:
cleaned_df.head()

,inspection_id,violation_code,violation_text,inspection_date,results,risk,dba_name,facility_type,address,zip
0,2638434,47,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,06/12/2026,Pass,Risk 1 (High),LITTLE VILAGE NURSING AND REHABILITATION CENTE...,LONG TERM CARE,2320 S LAWNDALE AVE,60623.0
1,2638421,NaN,,06/12/2026,Pass,Risk 2 (Medium),TAWNEY COFFEE LLC,RESTAURANT,3055-3057 W POPE JOHN PAUL II DR,60632.0
2,2638431,NaN,,06/12/2026,Pass,Risk 1 (High),BORELLI'S,RESTAURANT,2124 W LAWRENCE AVE,60625.0
3,2638417,NaN,,06/12/2026,Not Ready,Unknown,PINK HEALTH NUTRITION,Unknown,3015 W 63RD ST,60629.0
4,2638381,16,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,06/11/2026,Pass,Risk 1 (High),TAQUERIA LOS GALLOS INC,RESTAURANT,4209-4211 W 26TH ST,60623.0


## Feature Engineering
I want to make a year and month column for future analysis.

In [22]:
cleaned_df.dtypes

inspection_id       int64
violation_code        str
violation_text        str
inspection_date       str
results               str
risk                  str
dba_name              str
facility_type         str
address               str
zip                object
dtype: object

In [23]:
import datetime as dt

cleaned_df["year"] = cleaned_df["inspection_date"].apply(lambda x: dt.datetime.strptime(x, "%m/%d/%Y").year)
cleaned_df["month"] = cleaned_df["inspection_date"].apply(lambda x: dt.datetime.strptime(x, "%m/%d/%Y").month)

In [24]:
cleaned_df.head()

,inspection_id,violation_code,violation_text,inspection_date,results,risk,dba_name,facility_type,address,zip,year,month
0,2638434,47,47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE...,06/12/2026,Pass,Risk 1 (High),LITTLE VILAGE NURSING AND REHABILITATION CENTE...,LONG TERM CARE,2320 S LAWNDALE AVE,60623.0,2026,6
1,2638421,NaN,,06/12/2026,Pass,Risk 2 (Medium),TAWNEY COFFEE LLC,RESTAURANT,3055-3057 W POPE JOHN PAUL II DR,60632.0,2026,6
2,2638431,NaN,,06/12/2026,Pass,Risk 1 (High),BORELLI'S,RESTAURANT,2124 W LAWRENCE AVE,60625.0,2026,6
3,2638417,NaN,,06/12/2026,Not Ready,Unknown,PINK HEALTH NUTRITION,Unknown,3015 W 63RD ST,60629.0,2026,6
4,2638381,16,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,06/11/2026,Pass,Risk 1 (High),TAQUERIA LOS GALLOS INC,RESTAURANT,4209-4211 W 26TH ST,60623.0,2026,6


## Matching in Python

In order to make the analysis more applicable, I want to reduce the number of unique categories by grouping together similar facility types. For example, a restaurant and restaurant/bar can be grouped together and analyzed as restaurants.

In [25]:
cleaned_df["facility_type"].value_counts().head(100)

facility_type
RESTAURANT                      786596
GROCERY STORE                   140475
SCHOOL                           56974
CHILDREN'S SERVICES FACILITY     19377
BAKERY                           17921
                                 ...  
WRIGLEY ROOFTOP                     50
RIVERWALK                           49
GROCERY/BUTCHER                     49
PASTRY SCHOOL                       48
OUTREACH CULINARY KITCHEN           46
Name: count, Length: 100, dtype: int64

Looking through the value counts for the facility types, these are the most common facility types that I want to standardized the less common types into:
* RESTAURANT
* GROCERY STORE
* SCHOOL
* COMMUNITY BUILDINGS (CHILDREN'S SERVICES FACILITY, CHURCH ETC.)
* BAKERY
* HEALTHCARE (HOSPITAL, NURSING HOME ETC.)
* DAYCARE
* CATERING
* BAR
* LIQUOR
* MOBILE FOOD
* TAVERN
* BANQUET
* GAS STATION
* CAFE/COFFEE

In [26]:
patterns = {
    "RESTAURANT": "RESTAURANT",
    "DINER": "RESTAURANT",
    "GROCERY": "GROCERY",
    "CONVENIENCE": "GROCERY",
    "MART": "GROCERY",
    "BAKERY": "BAKERY",
    "TAVERN": "TAVERN",
    "BAR": "BAR",
    "LIQUOR": "LIQUOR",
    "CATERING": "CATERING",
    "BANQUET": "BANQUET",
    "MOBILE": "MOBILE FOOD",
    "SCHOOL": "SCHOOL",
    "DAYCARE": "DAYCARE",
    "CHILDREN": "COMMUNITY BUILDINGS",
    "SERVICE": "COMMUNITY BUILDINGS",
    "CHURCH": "COMMUNITY BUILDINGS",
    "SHELTER": "COMMUNITY BUILDINGS",
    "GAS": "GAS STATION",
    "CAFE": "CAFE",
    "COFFEE": "CAFE",
    "HOSPITAL": "HEALTHCARE",
    "NURSING": "HEALTHCARE",
    "CARE": "HEALTHCARE"
}

def standardize_facility_type(facility_type: str) -> str:
    """Takes in a facility type from the facility_type column, and attempts to match it to a larger category"""

    if pd.isna(facility_type):
        return None
    
    facility_type_cleaned = facility_type.upper().strip() # This is excessive but mostly a precaution in case, somehow the SQL cleaning did not work. Might also work on 'Unknown'

    for key, value in patterns.items():
        if key in facility_type_cleaned:
            return value
    
    return "OTHER"
    

cleaned_df["facility_type_clean"] = cleaned_df["facility_type"].apply(standardize_facility_type)

In [27]:
cleaned_df.loc[cleaned_df["facility_type_clean"] == "OTHER", "facility_type"].value_counts().head(50)

facility_type
Unknown                                   5798
WHOLESALE                                 1675
SPECIAL EVENT                              909
SHARED KITCHEN                             785
SHARED KITCHEN USER (LONG TERM)            693
LIVE POULTRY                               549
STADIUM                                    472
ROOFTOP                                    170
SUPPORTIVE LIVING                          148
COMMISSARY                                 145
HOTEL                                      141
ASSISTED LIVING                            134
ICE CREAM SHOP                             128
PALETERIA                                  128
KIOSK                                      119
POP-UP ESTABLISHMENT HOST-TIER II           98
PUBLIC SHCOOL                               98
THEATRE                                     93
THEATER                                     92
ROOFTOPS                                    86
NAVY PIER KIOSK                             84

## Double checking COALESCE and UPPER(TRIM())

In [28]:
cleaned_df["risk"].value_counts()

risk
Risk 1 (High)      859957
Risk 2 (Medium)    180491
Risk 3 (Low)        59019
All                   100
Unknown                97
Name: count, dtype: int64

In [29]:
cleaned_df["zip"].value_counts()

zip
60614.0    46355
60647.0    39621
60657.0    36559
60625.0    35101
60618.0    33580
           ...  
60478.0        1
60440.0        1
60022.0        1
60423.0        1
60155.0        1
Name: count, Length: 137, dtype: int64

In [30]:
cleaned_df["facility_type"].value_counts()

facility_type
RESTAURANT                      786596
GROCERY STORE                   140475
SCHOOL                           56974
CHILDREN'S SERVICES FACILITY     19377
BAKERY                           17921
                                 ...  
RELIGIOUS                            1
TAVERN/LIQUOR                        1
INCUBATOR                            1
WHOLESALE BAKERY                     1
KIDS CAFE'                           1
Name: count, Length: 473, dtype: int64

In [31]:
cleaned_df["dba_name"].value_counts()

dba_name
SUBWAY                                          12011
DUNKIN DONUTS                                    6735
MCDONALD'S                                       2983
7-ELEVEN                                         2338
MCDONALDS                                        1672
                                                ...  
GAS ON CALIFORNIA                                   1
SCRUB A DUB FULLERTON                               1
BUTTER                                              1
SURF & TURF WHERE SEAFOOD & STEAK MEET, INC.        1
MAKIA FOOD                                          1
Name: count, Length: 34481, dtype: int64

In [32]:
cleaned_df["address"].value_counts()

address
11601 W TOUHY AVE        11537
2300 S THROOP ST          1861
500 W MADISON ST          1796
5700 S CICERO AVE         1708
7601 S CICERO AVE         1304
                         ...  
700 N MILWAUKEE AVE          1
2152 E 71ST ST               1
1960 W 13 TH ST              1
13205 S MUSKEGON AVE         1
2458 S CALIFORNIA AVE        1
Name: count, Length: 20256, dtype: int64

## Exporting Clean Data

In [33]:
cleaned_df.to_csv("data\\clean\\food_inspections_cleaned.csv")